# Results

In [96]:
PROBLEMS_FILE = './data/problems.json'
SOLUTIONS_FILE = './data/solutions.json'
EVALUATIONS_FILE = './data/evaluations.json'

In [97]:
from src import read_json

problems = read_json(PROBLEMS_FILE)
solutions = read_json(SOLUTIONS_FILE)
evaluations = read_json(EVALUATIONS_FILE)

## Validation via models comparison

In [98]:
import pandas as pd

data = []

for evaluation_id, evaluation_data in evaluations.items():
    solution_id = evaluation_data['solution_id']
    solution_model = solutions[solution_id]['model']
    problem_id = solutions[solution_id]['problem_id']
    for evaluation_run,evaluation in enumerate(evaluation_data['content']):
        for criterion in evaluation.keys():
            score = evaluation[criterion]['score']
            data.append({
                'evaluation_id': evaluation_id,
                'model': solution_model,
                'problem_id': problem_id,
                'evaluation_run': evaluation_run,
                'criterion': criterion,
                'score': score
            })

df = pd.DataFrame(data)
df

,evaluation_id,model,problem_id,evaluation_run,criterion,score
0,da7baa97e894560786cae1c59694a37c,deepseek/deepseek-v4-pro,82fbc85fcfff5fe780b66e3914768cd3,0,framing,4
1,da7baa97e894560786cae1c59694a37c,deepseek/deepseek-v4-pro,82fbc85fcfff5fe780b66e3914768cd3,0,decomposition,4
2,da7baa97e894560786cae1c59694a37c,deepseek/deepseek-v4-pro,82fbc85fcfff5fe780b66e3914768cd3,0,diversity,3
3,da7baa97e894560786cae1c59694a37c,deepseek/deepseek-v4-pro,82fbc85fcfff5fe780b66e3914768cd3,0,coherence,5
4,da7baa97e894560786cae1c59694a37c,deepseek/deepseek-v4-pro,82fbc85fcfff5fe780b66e3914768cd3,0,justification,3
...,...,...,...,...,...,...
1195,8141a58540375a5cb724c326667dc0f7,microsoft/phi-4-mini-instruct,471d08ec78a55a42822be62273785153,4,coherence,3
1196,8141a58540375a5cb724c326667dc0f7,microsoft/phi-4-mini-instruct,471d08ec78a55a42822be62273785153,4,justification,2
1197,8141a58540375a5cb724c326667dc0f7,microsoft/phi-4-mini-instruct,471d08ec78a55a42822be62273785153,4,uncertainty_handling,2
1198,8141a58540375a5cb724c326667dc0f7,microsoft/phi-4-mini-instruct,471d08ec78a55a42822be62273785153,4,knowledge_integration,3


In [99]:
df.groupby(['model', 'criterion'])['score'].agg(lambda s : f'{s.mean().round(2)} ± {s.std().round(2)}').unstack(level=0)

model,deepseek/deepseek-v4-pro,microsoft/phi-4-mini-instruct,openai/gpt-4o-mini
criterion,,,
coherence,4.22 ± 0.42,3.42 ± 0.5,3.9 ± 0.3
decomposition,4.18 ± 0.39,3.16 ± 0.37,3.68 ± 0.47
diversity,3.14 ± 0.45,2.34 ± 0.48,2.46 ± 0.54
framing,4.14 ± 0.35,3.1 ± 0.3,3.36 ± 0.48
justification,3.18 ± 0.39,2.2 ± 0.4,2.56 ± 0.5
knowledge_integration,4.22 ± 0.42,2.98 ± 0.25,3.16 ± 0.37
metacognition,2.24 ± 0.43,1.1 ± 0.3,1.3 ± 0.46
uncertainty_handling,2.6 ± 0.73,1.42 ± 0.54,1.78 ± 0.79


In [100]:
df.groupby(['model']).agg({'score':lambda s : f'{s.mean().round(2)} ± {s.std().round(2)}'}).transpose()

model,deepseek/deepseek-v4-pro,microsoft/phi-4-mini-instruct,openai/gpt-4o-mini
score,3.49 ± 0.88,2.46 ± 0.89,2.78 ± 1.0


## ICC and Krippendorff's alpha 

In [104]:
import pandas as pd
import pingouin as pg

data = []

for evaluation_id, evaluation_data in evaluations.items():
    solution_id = evaluation_data['solution_id']
    solution_model = solutions[solution_id]['model']

    # ICC
    icc_data = []
    for evaluation_run,evaluation in enumerate(evaluation_data['content']):
        for criterion in evaluation.keys():
            score = evaluation[criterion]['score']
            icc_data.append({
                'judge': evaluation_run,
                'criterion': criterion,
                'score': score
            })
    icc_df = pg.intraclass_corr(pd.DataFrame(icc_data), targets='criterion', raters='judge', ratings='score')
    icc_dict = icc_df.set_index('Type')['ICC'].to_dict()
    data.append({
        'model': solution_model,
        **icc_dict
    })

    # Krippendorff's alpha
    

df = pd.DataFrame(data)
df.head()

/home/vasilstar/fp2mp-eval/.venv/lib/python3.10/site-packages/pingouin/parametric.py:1008: RuntimeWarning: divide by zero encountered in scalar divide
  fval = msbetween / mserror
/home/vasilstar/fp2mp-eval/.venv/lib/python3.10/site-packages/pingouin/reliability.py:361: RuntimeWarning: divide by zero encountered in scalar divide
  f1k = msb / msw
/home/vasilstar/fp2mp-eval/.venv/lib/python3.10/site-packages/pingouin/reliability.py:366: RuntimeWarning: divide by zero encountered in scalar divide
  f2k = f3k = msb / mse
/home/vasilstar/fp2mp-eval/.venv/lib/python3.10/site-packages/pingouin/reliability.py:387: RuntimeWarning: invalid value encountered in scalar divide
  l1 = (f1l - 1) / (f1l + (k - 1))
/home/vasilstar/fp2mp-eval/.venv/lib/python3.10/site-packages/pingouin/reliability.py:388: RuntimeWarning: invalid value encountered in scalar divide
  u1 = (f1u - 1) / (f1u + (k - 1))
/home/vasilstar/fp2mp-eval/.venv/lib/python3.10/site-packages/pingouin/reliability.py:391: RuntimeWarning:

,model,"ICC(1,1)","ICC(A,1)","ICC(C,1)","ICC(1,k)","ICC(A,k)","ICC(C,k)"
0,deepseek/deepseek-v4-pro,0.865986,0.866667,0.889251,0.969979,0.970149,0.975697
1,openai/gpt-4o-mini,0.783951,0.789790,0.913194,0.947761,0.949458,0.981343
2,microsoft/phi-4-mini-instruct,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
3,deepseek/deepseek-v4-pro,0.712441,0.712610,0.714706,0.925305,0.925362,0.926067
4,openai/gpt-4o-mini,0.755089,0.763441,0.920370,0.939082,0.941645,0.982991


In [102]:
df.groupby('model').agg(lambda s : f'{s.mean().round(2)} ± {s.std().round(2)}')

,"ICC(1,1)","ICC(A,1)","ICC(C,1)","ICC(1,k)","ICC(A,k)","ICC(C,k)"
model,,,,,,
deepseek/deepseek-v4-pro,0.82 ± 0.1,0.83 ± 0.09,0.88 ± 0.07,0.96 ± 0.03,0.96 ± 0.03,0.97 ± 0.02
microsoft/phi-4-mini-instruct,0.85 ± 0.09,0.85 ± 0.08,0.88 ± 0.08,0.96 ± 0.02,0.97 ± 0.02,0.97 ± 0.02
openai/gpt-4o-mini,0.79 ± 0.07,0.8 ± 0.07,0.87 ± 0.06,0.95 ± 0.02,0.95 ± 0.02,0.97 ± 0.02


In [103]:
df.drop(columns=['model']).apply(lambda s : f'{s.mean().round(2)} ± {s.std().round(2)}')

ICC(1,1)    0.82 ± 0.08
ICC(A,1)    0.83 ± 0.08
ICC(C,1)    0.87 ± 0.07
ICC(1,k)    0.96 ± 0.02
ICC(A,k)    0.96 ± 0.02
ICC(C,k)    0.97 ± 0.02
dtype: object